# Phase 5 — Evidence-Cited RAG Validation

Question → cross-stream retrieval → Claude synthesizes an answer where **every claim cites its source** ([n] → code `path:line`, commit SHA, or issue #).

**Needs `ANTHROPIC_API_KEY` in .env** for the generation step. Retrieval + prompt assembly work without it.

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from archaeologist.rag import prompts
from archaeologist.rag.llm import has_api_key
from archaeologist.rag.pipeline import answer_question, retrieve
print("ANTHROPIC_API_KEY set:", has_api_key())

## Step 1 — retrieve cross-stream evidence

In [ ]:
Q = "why does Flask use an application context and when is it pushed"
evidence = retrieve(Q, k=8)
for i, e in enumerate(evidence, 1):
    print(f"  [{i}] ({e['stream']:6}) {e['citation']:26.26} {e['title'][:44]}")

## Step 2 — the grounded prompt (preview)

In [ ]:
prompt = prompts.build_prompt(Q, evidence)
print(f"system: {len(prompts.SYSTEM)} chars | prompt: {len(prompt)} chars\n")
print(prompt[:700])

## Step 3 — the cited answer
Runs the full pipeline. Without a key you'll see a placeholder + the evidence.

In [ ]:
result = answer_question(Q, k=8)
print(result.answer)

In [ ]:
# 'why' questions that should pull in commits/issues, not just code:
for q in ["what changed about async view support and why",
          "what would break if I removed the application context"]:
    print(f"\n{'=' * 70}\nQ: {q}\n")
    print(answer_question(q, k=8).answer)

## Summary

In [ ]:
print("Phase 5 —", "RAG OK ✅" if has_api_key() else "RETRIEVAL OK ⚠️ (add ANTHROPIC_API_KEY to generate)")
print("  retrieval + prompt assembly: working")
print("  generation:", "working" if has_api_key() else "needs ANTHROPIC_API_KEY")